# Getting the most out of GPT-5.4 for document and multimodal understanding

GPT-5.4 is a major step forward for real-world multimodal workloads.

Documents that previously broke vision models or required multiple systems to do OCR/ layout detection/custom parsing, such as dense scans, handwritten forms, engineering diagrams, and chart-heavy reports, can now be read and reasoned over reliably in a single model pass with GPT-5.4. 

However, model configuration is key for unlocking SOTA results. Small choices around image detail, verbosity, reasoning effort, and tool usage can significantly affect performance. This notebook walks through practical patterns for configuring robust multimodal pipelines with GPT-5.4.

## A concrete launch takeaway

While drafting this notebook on March 4, 2026, I ran a small known-answer slice with the exact bundled assets and prompts below. The current GPT-5.4 preview (`galapagos-alpha`) matched or beat `gpt-5` on every task and returned faster on all three:

- Handwritten form extraction with `detail="original"`: `gpt-5` scored `12/13` fields in `22.66s`; `galapagos-alpha` scored `13/13` in `8.75s`.
- Floorplan reasoning: both models scored `6/6`, but `galapagos-alpha` returned in `3.11s` versus `7.12s` for `gpt-5`.
- Structured invoice extraction: both models scored `7/7`, but `galapagos-alpha` returned in `5.09s` versus `16.16s` for `gpt-5`.

Treat that as a reproducible notebook slice, not a benchmark paper. The point is to make the launch story concrete and testable on bundled assets, not to over-claim from one run.


![OmniDocBench selected runs](../../images/omnidocbench_selected_runs.png)


## A quick decision guide

| If your task looks like this | Start with this setup | Why |
|---|---|---|
| Ordinary document QA or extraction | `detail="auto"` | Lowest-friction default for readable pages |
| Dense scans, screenshots, handwriting, or tiny labels | `detail="original"` | Preserves the small visual evidence that gets lost first |
| Literal transcription or markdown conversion | `text={"verbosity": "high"}` | Pushes the model to keep more layout and fewer paraphrases |
| Benchmark and eval runs | Keep any sampling controls your model exposes fixed | Reduces variance and keeps comparisons fair |
| Region localization | Ask for `[x_min, y_min, x_max, y_max]` in a fixed `0..999` grid | Easy to crop, draw, debug, and feed into downstream systems |
| Chart, table, form, or drawing QA across multiple regions | `reasoning={"effort": "high"}` | Helps when the answer depends on combining evidence from different parts of the image |
| Multi-pass visual inspection | Add Code Interpreter | Best when a human would zoom, crop, rotate, or inspect several subregions before answering |
| Restricted environment without Code Interpreter | Localize -> crop -> rerun | Narrower, cheaper, and easier to control than general Python execution |


## Setup

Install the minimum dependencies if needed:

```bash
pip install --upgrade openai pandas pillow
```

The notebook uses these bundled assets:

- a dense hotel invoice for transcription fidelity
- a handwritten insurance application for detail tuning
- a police report form for localization and tool-assisted inspection
- a synthetic apartment floorplan for building-plan-style reasoning with known answers
- a tournament bracket image for dense long-range layout reasoning
- a UML-style agent architecture diagram for developer-facing technical-diagram QA
- a line chart for chart QA

The Code Interpreter and benchmark sections are opt-in because they are more expensive and can add noticeable latency.


In [ ]:
import base64
import json
import mimetypes
import os
import re
import tempfile
import time
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from PIL import Image, ImageDraw
from openai import APIConnectionError, OpenAI


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "registry.yaml").exists() and (candidate / "examples").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the openai-cookbook repo root from the current working directory."
    )


REPO_ROOT = find_repo_root()

ASSETS = {
    "benchmark_figure": REPO_ROOT / "images/omnidocbench_selected_runs.png",
    "invoice": REPO_ROOT / "images/sample_hotel_invoice.png",
    "handwritten_form": REPO_ROOT / "images/3C_insurance_form.png",
    "police_form": REPO_ROOT / "examples/multimodal/images/police_form.png",
    "floorplan": REPO_ROOT / "examples/multimodal/images/apartment_floorplan.png",
    "bracket": REPO_ROOT / "examples/multimodal/images/bracket.png",
    "uml_diagram": REPO_ROOT / "images/oo_aa_image_2_uml_diagram.png",
    "chart": REPO_ROOT / "images/NotRealCorp_chart.png",
}

INVOICE_GROUND_TRUTH_PATH = (
    REPO_ROOT
    / "examples/data/hotel_invoices/transformed_invoice_json/transformed_premierinn_GABCI19014325_extracted.json"
)

for asset_path in [*ASSETS.values(), INVOICE_GROUND_TRUTH_PATH]:
    if not asset_path.exists():
        raise FileNotFoundError(f"Missing bundled asset: {asset_path}")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY before running this notebook.")

# Keep the pre-release name here so the notebook runs internally now.
# Swap this to the launch-approved public slug before publishing if needed.
PRIMARY_MODEL = os.environ.get("OPENAI_MULTIMODAL_MODEL", "galapagos-alpha")
BASELINE_MODEL = os.environ.get("OPENAI_MULTIMODAL_BASELINE_MODEL", "gpt-5")
COMPARISON_MODELS = list(
    dict.fromkeys(
        model_name.strip()
        for model_name in os.environ.get(
            "OPENAI_MULTIMODAL_COMPARISON_MODELS",
            f"{BASELINE_MODEL},{PRIMARY_MODEL}",
        ).split(",")
        if model_name.strip()
    )
)

RUN_OPTIONAL_CODE_INTERPRETER = False
RUN_OPTIONAL_BENCHMARKS = False

client = OpenAI(api_key=OPENAI_API_KEY)

invoice_ground_truth = json.loads(INVOICE_GROUND_TRUTH_PATH.read_text())

HANDWRITING_EXPECTED = {
    "applicant_name": "Smith, James L",
    "applicant_email": "jsmith1@gmail.com",
    "applicant_home_phone": "510 331 5555",
    "applicant_cell_phone": "510 212 5555",
    "co_applicant_name": "Roberts, Jesse T",
    "co_applicant_email": "jrobertsjr@gmail.com",
    "co_applicant_home_phone": "510 331 5555",
    "co_applicant_work_phone": "415 626 5555",
    "effective_date": "5/31/25",
    "expiration_date": "5/31/27",
    "dwelling_coverage_limit_usd": 900000,
    "square_footage": 1200,
    "year_of_construction": 2005,
}

FLOORPLAN_EXPECTED = {
    "total_named_rooms_excluding_hallways_and_closets": 7,
    "largest_room": "Living Room",
    "room_immediately_east_of_kitchen": "Dining",
    "room_immediately_south_of_study": "Bedroom 2",
    "overall_width_ft": 36,
    "overall_height_ft": 24,
}

UML_EXPECTED = {
    "central_class": "BaseAgent",
    "classes_directly_used_by_baseagent": [
        "ToolManager",
        "LanguageModelInterface",
        "ChatMessages",
    ],
    "component_that_manages_tool_definitions": "ToolManager",
    "implementation_of_language_model_interface": "OpenAILanguageModel",
    "factory_class_for_openai_client": "OpenAIClientFactory",
    "relationship_from_toolmanager_to_toolinterface": "manages",
}

INVOICE_EXPECTED = {
    "hotel_name": invoice_ground_truth["hotel_information"]["name"],
    "check_in_date": invoice_ground_truth["invoice_information"]["check_in_date"],
    "check_out_date": invoice_ground_truth["invoice_information"]["check_out_date"],
    "room_number": invoice_ground_truth["invoice_information"]["room_number"],
    "total_gross_eur": invoice_ground_truth["totals_summary"]["total_gross"],
    "balance_due_eur": invoice_ground_truth["totals_summary"]["balance_due"],
    "breakfast_line_items_count": sum(
        1
        for charge in invoice_ground_truth["charges"]
        if "breakfast" in charge["description"].lower()
    ),
}

FIELD_TOLERANCES = {
    "dwelling_coverage_limit_usd": 0,
    "square_footage": 0,
    "year_of_construction": 0,
    "overall_width_ft": 0,
    "overall_height_ft": 0,
    "total_gross_eur": 0.01,
    "balance_due_eur": 0.01,
    "breakfast_line_items_count": 0,
}

pd.Series(
    {
        "repo_root": str(REPO_ROOT),
        "primary_model": PRIMARY_MODEL,
        "baseline_model": BASELINE_MODEL,
        "comparison_models": ", ".join(COMPARISON_MODELS),
        "run_optional_code_interpreter": RUN_OPTIONAL_CODE_INTERPRETER,
        "run_optional_benchmarks": RUN_OPTIONAL_BENCHMARKS,
    }
)


In [ ]:
def as_path(asset_ref: str | Path) -> Path:
    if isinstance(asset_ref, Path):
        return asset_ref
    if asset_ref in ASSETS:
        return ASSETS[asset_ref]
    path = Path(asset_ref)
    if path.exists():
        return path
    raise FileNotFoundError(f"Unknown asset reference: {asset_ref}")


def image_to_data_url(asset_ref: str | Path) -> str:
    path = as_path(asset_ref)
    mime_type = mimetypes.guess_type(path.name)[0] or "image/png"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


def run_vision_request(
    prompt: str,
    asset_ref: str | Path,
    *,
    model: str = PRIMARY_MODEL,
    detail: str = "auto",
    verbosity: str | None = None,
    reasoning_effort: str | None = None,
    instructions: str | None = None,
    tools: list | None = None,
    include: list[str] | None = None,
    temperature: float | None = None,
    max_output_tokens: int | None = None,
    retries: int = 3,
    backoff_seconds: float = 2.0,
):
    request = {
        "model": model,
        "input": [
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {
                        "type": "input_image",
                        "image_url": image_to_data_url(asset_ref),
                        "detail": detail,
                    },
                ],
            }
        ],
    }

    if verbosity is not None:
        request["text"] = {"verbosity": verbosity}
    if reasoning_effort is not None:
        request["reasoning"] = {"effort": reasoning_effort}
    if instructions is not None:
        request["instructions"] = instructions
    if tools is not None:
        request["tools"] = tools
    if include is not None:
        request["include"] = include
    if temperature is not None:
        request["temperature"] = temperature
    if max_output_tokens is not None:
        request["max_output_tokens"] = max_output_tokens

    for attempt in range(1, retries + 1):
        try:
            return client.responses.create(**request)
        except APIConnectionError:
            if attempt == retries:
                raise
            time.sleep(backoff_seconds * attempt)


def response_usage(response) -> dict:
    usage = getattr(response, "usage", None)
    if usage is None:
        return {}
    return usage.model_dump() if hasattr(usage, "model_dump") else dict(usage)


def extract_json(text: str):
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)

    candidates = [cleaned]
    for opening, closing in [("{", "}"), ("[", "]")]:
        start = cleaned.find(opening)
        end = cleaned.rfind(closing)
        if start != -1 and end != -1 and end > start:
            candidates.insert(0, cleaned[start : end + 1])

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            continue

    raise ValueError("Response did not contain valid JSON.")


def maybe_parse_json(text: str):
    try:
        return extract_json(text)
    except ValueError:
        return None


def normalize_text(value):
    if value is None:
        return None
    text = unicodedata.normalize("NFKD", str(value))
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text).strip()
    return text


def values_match(predicted, expected, tolerance: float | None = None) -> bool:
    if isinstance(expected, list):
        if not isinstance(predicted, list):
            predicted = [] if predicted is None else [predicted]
        left = sorted(normalize_text(item) for item in predicted if item is not None)
        right = sorted(normalize_text(item) for item in expected if item is not None)
        return left == right

    if isinstance(expected, (int, float)):
        try:
            predicted_float = float(predicted)
        except (TypeError, ValueError):
            return False
        tolerance = 0 if tolerance is None else tolerance
        return abs(predicted_float - float(expected)) <= tolerance

    return normalize_text(predicted) == normalize_text(expected)


def score_json_response(predicted: dict | None, expected: dict, tolerances: dict | None = None):
    tolerances = tolerances or {}
    rows = []
    for field, expected_value in expected.items():
        predicted_value = None if predicted is None else predicted.get(field)
        match = values_match(predicted_value, expected_value, tolerances.get(field))
        rows.append(
            {
                "field": field,
                "expected": expected_value,
                "predicted": predicted_value,
                "match": match,
            }
        )
    frame = pd.DataFrame(rows)
    accuracy = frame["match"].mean() if not frame.empty else 0.0
    return frame, accuracy


def count_anchor_hits(text: str, anchors: list[str]) -> pd.DataFrame:
    normalized_text = normalize_text(text) or ""
    rows = []
    for anchor in anchors:
        rows.append(
            {
                "anchor": anchor,
                "present": (normalize_text(anchor) or "") in normalized_text,
            }
        )
    return pd.DataFrame(rows)


def preview_text(title: str, text: str, max_chars: int = 1800):
    clipped = text if len(text) <= max_chars else text[:max_chars] + "\n\n...[truncated]..."
    display(Markdown(f"### {title}\n\n```text\n{clipped}\n```"))


def count_code_interpreter_calls(response) -> int:
    output_items = getattr(response, "output", None) or []
    count = 0
    for item in output_items:
        item_type = item.get("type") if isinstance(item, dict) else getattr(item, "type", None)
        if item_type == "code_interpreter_call":
            count += 1
    return count


def denormalize_bbox(bbox: list[int], width: int, height: int) -> list[int]:
    x_min, y_min, x_max, y_max = bbox
    return [
        round(x_min * (width - 1) / 999),
        round(y_min * (height - 1) / 999),
        round(x_max * (width - 1) / 999),
        round(y_max * (height - 1) / 999),
    ]


def overlay_bboxes(asset_ref: str | Path, items: list[dict]):
    image = Image.open(as_path(asset_ref)).convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    palette = ["red", "dodgerblue", "limegreen", "orange", "magenta", "cyan"]

    for index, item in enumerate(items):
        color = palette[index % len(palette)]
        pixel_bbox = denormalize_bbox(item["bbox"], width, height)
        draw.rectangle(pixel_bbox, outline=color, width=4)
        draw.text((pixel_bbox[0] + 4, max(0, pixel_bbox[1] - 16)), item["label"], fill=color)

    return image


def crop_from_normalized_bbox(
    asset_ref: str | Path,
    bbox: list[int],
    *,
    padding_ratio: float = 0.04,
) -> Path:
    source = Image.open(as_path(asset_ref)).convert("RGB")
    width, height = source.size
    x_min, y_min, x_max, y_max = denormalize_bbox(bbox, width, height)

    pad_x = round((x_max - x_min) * padding_ratio)
    pad_y = round((y_max - y_min) * padding_ratio)

    crop = source.crop(
        (
            max(0, x_min - pad_x),
            max(0, y_min - pad_y),
            min(width, x_max + pad_x),
            min(height, y_max + pad_y),
        )
    )

    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    tmp_path = Path(tmp.name)
    tmp.close()
    crop.save(tmp_path)
    return tmp_path


def run_eval_case(
    name: str,
    prompt: str,
    asset_ref: str | Path,
    expected: dict,
    *,
    model: str,
    detail: str = "auto",
    verbosity: str | None = None,
    reasoning_effort: str | None = None,
):
    started_at = time.perf_counter()
    response = run_vision_request(
        prompt,
        asset_ref,
        model=model,
        detail=detail,
        verbosity=verbosity,
        reasoning_effort=reasoning_effort,
    )
    elapsed_s = round(time.perf_counter() - started_at, 2)
    parsed = maybe_parse_json(response.output_text)
    score_frame, accuracy = score_json_response(parsed, expected, FIELD_TOLERANCES)
    usage = response_usage(response)
    return {
        "name": name,
        "model": model,
        "detail": detail,
        "verbosity": verbosity or "default",
        "reasoning": reasoning_effort or "default",
        "elapsed_s": elapsed_s,
        "field_accuracy": round(float(accuracy), 3),
        "matches": int(score_frame["match"].sum()),
        "total_fields": int(len(score_frame)),
        "output_tokens": usage.get("output_tokens"),
        "parsed": parsed,
        "score_frame": score_frame,
        "response": response,
    }


## 1. Raise image detail for dense pages and handwriting

`detail="auto"` is still the right default for ordinary pages. Move to `detail="original"` when the page is dense, handwritten, low contrast, or full of small labels. That is the first knob to reach for when the model is almost right but keeps missing small visual evidence.

This example intentionally includes small email and phone fields, not just the larger handwritten names. Those are the kinds of details that tend to degrade first when the image is downsampled.

![Handwritten insurance form](../../images/3C_insurance_form.png)


In [ ]:
handwriting_prompt = """
Read the handwritten earthquake insurance application and return JSON with these keys:
- applicant_name
- applicant_email
- applicant_home_phone
- applicant_cell_phone
- co_applicant_name
- co_applicant_email
- co_applicant_home_phone
- co_applicant_work_phone
- effective_date
- expiration_date
- dwelling_coverage_limit_usd
- square_footage
- year_of_construction

Rules:
- Copy names and emails exactly as written.
- Return phone numbers as plain strings with spaces preserved if visible.
- Return dates exactly as written.
- Normalize currency to an integer number of USD with no commas or dollar signs.
- Return integers for square footage and year.
- Return JSON only.
"""

handwriting_results = []
for detail in ["auto", "original"]:
    started_at = time.perf_counter()
    response = run_vision_request(
        handwriting_prompt,
        "handwritten_form",
        model=PRIMARY_MODEL,
        detail=detail,
    )
    elapsed_s = round(time.perf_counter() - started_at, 2)
    parsed = maybe_parse_json(response.output_text)
    score_frame, accuracy = score_json_response(parsed, HANDWRITING_EXPECTED, FIELD_TOLERANCES)
    usage = response_usage(response)
    handwriting_results.append(
        {
            "detail": detail,
            "elapsed_s": elapsed_s,
            "field_accuracy": round(float(accuracy), 3),
            "matches": int(score_frame["match"].sum()),
            "total_fields": int(len(score_frame)),
            "output_tokens": usage.get("output_tokens"),
            "parsed": parsed,
            "score_frame": score_frame,
            "raw_output": response.output_text,
        }
    )

display(
    pd.DataFrame(
        [
            {
                key: result[key]
                for key in ["detail", "elapsed_s", "field_accuracy", "matches", "total_fields", "output_tokens"]
            }
            for result in handwriting_results
        ]
    )
)

for result in handwriting_results:
    display(Markdown(f"### {result['detail']} detail: parsed output"))
    if result["parsed"] is not None:
        print(json.dumps(result["parsed"], indent=2))
    else:
        preview_text(f"{result['detail']} detail raw output", result["raw_output"])
    display(result["score_frame"])


## 2. Turn verbosity high when transcription fidelity matters

Multimodal models naturally compress when you ask them to transcribe a page: they keep the meaning, but they often simplify whitespace, line breaks, and table-like layout. That is usually fine for QA. It is not fine when you need a layout-faithful markdown transcript.

`text={"verbosity": "high"}` is the simplest way to push the model toward a more literal rendering. Use it for OCR-style workloads, invoice conversion, and benchmark runs where completeness matters more than brevity.

To make the difference concrete, the cell below compares the two transcripts using a stricter set of visible anchors, including an exact booking identifier that one setting can miss.

![Hotel invoice](../../images/sample_hotel_invoice.png)


In [ ]:
invoice_prompt = """
Transcribe this hotel invoice into markdown.

Requirements:
- Preserve section headers, dates, invoice identifiers, line items, tax rows, and totals.
- Keep each visible line item on its own line.
- Do not summarize, rewrite, or normalize the content.
"""

invoice_anchors = [
    "Hamburg City (Zentrum)",
    "Rechnungskopie",
    "Herr Jens Walter",
    "GABCI19014325",
    "GABR15867",
    "550,60",
    "Premier Inn Frühstücksbuffet",
    "626",
]

invoice_runs = []
for label, verbosity in [("default", None), ("high", "high")]:
    started_at = time.perf_counter()
    response = run_vision_request(
        invoice_prompt,
        "invoice",
        model=PRIMARY_MODEL,
        detail="original",
        verbosity=verbosity,
        max_output_tokens=2500,
    )
    elapsed_s = round(time.perf_counter() - started_at, 2)
    anchor_frame = count_anchor_hits(response.output_text, invoice_anchors)
    usage = response_usage(response)
    invoice_runs.append(
        {
            "verbosity": label,
            "elapsed_s": elapsed_s,
            "anchor_coverage": round(float(anchor_frame["present"].mean()), 3),
            "anchors_found": int(anchor_frame["present"].sum()),
            "anchors_total": int(len(anchor_frame)),
            "output_tokens": usage.get("output_tokens"),
            "text": response.output_text,
            "anchor_frame": anchor_frame,
        }
    )

display(
    pd.DataFrame(
        [
            {
                key: result[key]
                for key in [
                    "verbosity",
                    "elapsed_s",
                    "anchor_coverage",
                    "anchors_found",
                    "anchors_total",
                    "output_tokens",
                ]
            }
            for result in invoice_runs
        ]
    )
)

preview_text("Default verbosity transcription preview", invoice_runs[0]["text"])
preview_text("High verbosity transcription preview", invoice_runs[1]["text"])

for result in invoice_runs:
    display(Markdown(f"### {result['verbosity']} verbosity anchor coverage"))
    display(result["anchor_frame"])


## 3. Ask for bounding boxes in a stable format

When you need localization, do not leave the coordinate format implicit. Ask for a fixed schema like `[x_min, y_min, x_max, y_max]` and a fixed coordinate space such as `0..999` with the origin in the top-left corner.

That makes the output easy to crop, draw, debug, compare across models, and feed into downstream systems.

![Police report form](images/police_form.png)


In [ ]:
bbox_prompt = """
Find the vehicle travel direction and damaged area codes for Vehicles 1 and 2 in this police report form.

Return JSON with this schema:
[
  {"label": "vehicle_1_travel_direction", "bbox": [x_min, y_min, x_max, y_max]},
  {"label": "vehicle_1_damaged_area_code", "bbox": [x_min, y_min, x_max, y_max]},
  {"label": "vehicle_2_travel_direction", "bbox": [x_min, y_min, x_max, y_max]},
  {"label": "vehicle_2_damaged_area_code", "bbox": [x_min, y_min, x_max, y_max]}
]

Use discrete normalized coordinates between 0 and 999.
Return JSON only.
"""

bbox_response = run_vision_request(
    bbox_prompt,
    "police_form",
    model=PRIMARY_MODEL,
    detail="original",
)
bbox_results = maybe_parse_json(bbox_response.output_text)
if bbox_results is None:
    raise ValueError(bbox_response.output_text)

display(pd.DataFrame(bbox_results))
display(overlay_bboxes("police_form", bbox_results))


## 4. Raise reasoning effort when the image is readable but the answer is compositional

Once the image is readable, the next bottleneck is often reasoning instead of perception. That happens with charts, tables, forms, and technical drawings where the answer depends on evidence from several different parts of the page.

The floorplan below is a stand-in for a more complex architectural or CAD-style example. It has enough structure to show the tradeoff clearly while still giving us known answers.

One practical lesson from this kind of benchmark is that higher reasoning is not free. If the default pass already solves the task, turning reasoning up just adds latency. Use it selectively.

![Apartment floorplan](images/apartment_floorplan.png)


In [ ]:
floorplan_prompt = """
Inspect this apartment floorplan and return JSON with these keys:
- total_named_rooms_excluding_hallways_and_closets
- largest_room
- room_immediately_east_of_kitchen
- room_immediately_south_of_study
- overall_width_ft
- overall_height_ft

Rules:
- Use the room labels and dimension annotations that are visible on the drawing.
- Return integers for numeric fields.
- Return JSON only.
"""

floorplan_results = []
for reasoning_effort in [None, "high"]:
    started_at = time.perf_counter()
    response = run_vision_request(
        floorplan_prompt,
        "floorplan",
        model=PRIMARY_MODEL,
        detail="original",
        reasoning_effort=reasoning_effort,
    )
    elapsed_s = round(time.perf_counter() - started_at, 2)
    parsed = maybe_parse_json(response.output_text)
    score_frame, accuracy = score_json_response(parsed, FLOORPLAN_EXPECTED, FIELD_TOLERANCES)
    usage = response_usage(response)
    floorplan_results.append(
        {
            "reasoning": reasoning_effort or "default",
            "elapsed_s": elapsed_s,
            "field_accuracy": round(float(accuracy), 3),
            "matches": int(score_frame["match"].sum()),
            "total_fields": int(len(score_frame)),
            "output_tokens": usage.get("output_tokens"),
            "parsed": parsed,
            "score_frame": score_frame,
            "raw_output": response.output_text,
        }
    )

display(
    pd.DataFrame(
        [
            {
                key: result[key]
                for key in ["reasoning", "elapsed_s", "field_accuracy", "matches", "total_fields", "output_tokens"]
            }
            for result in floorplan_results
        ]
    )
)

for result in floorplan_results:
    display(Markdown(f"### {result['reasoning']} reasoning"))
    if result["parsed"] is not None:
        print(json.dumps(result["parsed"], indent=2))
    else:
        preview_text(f"{result['reasoning']} reasoning raw output", result["raw_output"])
    display(result["score_frame"])


### Additional chart example

The same reasoning pattern helps with chart QA. If the answer depends on reading multiple series, comparing adjacent intervals, and estimating values rather than extracting a single label, raise reasoning effort before you reach for a tool.

![Line chart](../../images/NotRealCorp_chart.png)


In [ ]:
chart_prompt = """
Inspect this line chart and return JSON with these keys:
- largest_qoq_increase: {"channel": ..., "from_quarter": ..., "to_quarter": ..., "approx_delta_millions": ...}
- largest_qoq_drop: {"channel": ..., "from_quarter": ..., "to_quarter": ..., "approx_delta_millions": ...}
- fastest_growing_channel_overall

Rules:
- Use approximate values only when exact values are not printed.
- Base the answer on the visible lines and quarter labels.
- Return JSON only.
"""

chart_response = run_vision_request(
    chart_prompt,
    "chart",
    model=PRIMARY_MODEL,
    detail="original",
    reasoning_effort="high",
)
chart_json = maybe_parse_json(chart_response.output_text)
if chart_json is None:
    raise ValueError(chart_response.output_text)

print(json.dumps(chart_json, indent=2))


### Additional tournament bracket example

Dense tournament brackets are a good stress test for long-range visual reasoning. The model has to read mirrored trees, keep the left and right halves separate, then use the central championship score boxes to answer the actual question.

This is the example your colleague added upstream. In the live runs I checked while merging, both `gpt-5` and `galapagos-alpha` recovered the same champions, but `gpt-5` tended to preserve seed numbers in the team strings and took much longer on the same prompt.

![Tournament bracket](images/bracket.png)


In [ ]:
bracket_prompt = """
Inspect this tournament bracket image and return JSON with these keys:
- left_bracket_title
- right_bracket_title
- mens_champion_team
- womens_champion_team
- mens_runner_up_team
- womens_runner_up_team

Rules:
- Use the visible central championship score boxes.
- Team labels may include seeds; if you include a seed, keep it attached to the same string.
- Return JSON only.
"""

bracket_runs = []
for model in COMPARISON_MODELS:
    started_at = time.perf_counter()
    response = run_vision_request(
        bracket_prompt,
        "bracket",
        model=model,
        detail="original",
    )
    elapsed_s = round(time.perf_counter() - started_at, 2)
    parsed = maybe_parse_json(response.output_text)
    usage = response_usage(response)
    bracket_runs.append(
        {
            "model": model,
            "elapsed_s": elapsed_s,
            "output_tokens": usage.get("output_tokens"),
            "parsed": parsed,
            "raw_output": response.output_text,
        }
    )

display(
    pd.DataFrame(
        [
            {
                key: result[key]
                for key in ["model", "elapsed_s", "output_tokens"]
            }
            for result in bracket_runs
        ]
    )
)

for result in bracket_runs:
    display(Markdown(f"### {result['model']} bracket output"))
    if result["parsed"] is not None:
        print(json.dumps(result["parsed"], indent=2, ensure_ascii=False))
    else:
        preview_text(f"{result['model']} bracket raw output", result["raw_output"])


### Additional developer-facing technical diagram example

If your users work with architecture diagrams, UML, ERDs, or infra sketches, treat them like technical drawings rather than plain images. The model has to read class names, connector labels, and directionality, then compose them into a structured answer.

This example uses a UML-style agent architecture diagram and asks for relationships that matter to a developer, not just raw OCR.

![UML architecture diagram](../../images/oo_aa_image_2_uml_diagram.png)


In [ ]:
uml_prompt = """
Inspect this UML-style architecture diagram and return JSON with these keys:
- central_class
- classes_directly_used_by_baseagent
- component_that_manages_tool_definitions
- implementation_of_language_model_interface
- factory_class_for_openai_client
- relationship_from_toolmanager_to_toolinterface

Rules:
- Use the class names exactly as shown in the diagram.
- Return `classes_directly_used_by_baseagent` as an array.
- For the relationship field, use the connector label if visible.
- Return JSON only.
"""

uml_started = time.perf_counter()
uml_response = run_vision_request(
    uml_prompt,
    "uml_diagram",
    model=PRIMARY_MODEL,
    detail="original",
)
uml_elapsed = round(time.perf_counter() - uml_started, 2)
uml_json = maybe_parse_json(uml_response.output_text)
if uml_json is None:
    raise ValueError(uml_response.output_text)

uml_score_frame, uml_accuracy = score_json_response(uml_json, UML_EXPECTED)

usage = response_usage(uml_response)
display(
    pd.DataFrame(
        [
            {
                "elapsed_s": uml_elapsed,
                "field_accuracy": round(float(uml_accuracy), 3),
                "matches": int(uml_score_frame["match"].sum()),
                "total_fields": int(len(uml_score_frame)),
                "output_tokens": usage.get("output_tokens"),
            }
        ]
    )
)

print(json.dumps(uml_json, indent=2))
display(uml_score_frame)


## 5. Use Code Interpreter when the task needs a real second look

Native vision is often enough. Code Interpreter becomes useful when a human would open the image in another tab, zoom into several regions, rotate something, or sanity-check a reading before answering.

Reach for Code Interpreter when:

- the page is dense and the relevant evidence is scattered across multiple regions
- you expect to crop, zoom, rotate, or inspect intermediate views
- qualitative accuracy matters more than minimum latency

While drafting this notebook on March 4, 2026, the tool-assisted version of this police-form task took about `104.8s` and made `2` Code Interpreter calls. That is useful evidence, but it is not a generic speed story. Treat Code Interpreter here as a multi-pass inspection tool with an explicit latency tradeoff, not a free performance win.

The cell below compares a native pass with a tool-assisted pass on the same police-form question. This is opt-in because tool calls are slower and more expensive than a single native vision pass.


In [ ]:
ci_prompt = """
Read the police report form and return JSON with these keys:
- vehicle_1_travel_direction
- vehicle_1_damaged_area_code
- vehicle_2_travel_direction
- vehicle_2_damaged_area_code

Rules:
- If the form is hard to read, inspect the relevant regions before answering.
- Return JSON only.
"""

if RUN_OPTIONAL_CODE_INTERPRETER:
    native_started = time.perf_counter()
    native_response = run_vision_request(
        ci_prompt,
        "police_form",
        model=PRIMARY_MODEL,
        detail="original",
        reasoning_effort="high",
    )
    native_elapsed = round(time.perf_counter() - native_started, 2)

    with as_path("police_form").open("rb") as asset_file:
        uploaded_file = client.files.create(file=asset_file, purpose="user_data")

    ci_started = time.perf_counter()
    ci_response = client.responses.create(
        model=PRIMARY_MODEL,
        instructions=(
            "You are an expert document analyst. Use Code Interpreter before answering. "
            "Inspect the uploaded file, crop or zoom if needed, then answer in JSON."
        ),
        tools=[
            {
                "type": "code_interpreter",
                "container": {
                    "type": "auto",
                    "memory_limit": "4g",
                    "file_ids": [uploaded_file.id],
                },
            }
        ],
        include=["code_interpreter_call.outputs"],
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": ci_prompt + " Use the tool at least once before answering."},
                    {
                        "type": "input_image",
                        "image_url": image_to_data_url("police_form"),
                        "detail": "original",
                    },
                ],
            }
        ],
    )
    ci_elapsed = round(time.perf_counter() - ci_started, 2)

    comparison_df = pd.DataFrame(
        [
            {
                "mode": "native vision",
                "elapsed_s": native_elapsed,
                "code_interpreter_calls": 0,
                "output_tokens": response_usage(native_response).get("output_tokens"),
            },
            {
                "mode": "code interpreter",
                "elapsed_s": ci_elapsed,
                "code_interpreter_calls": count_code_interpreter_calls(ci_response),
                "output_tokens": response_usage(ci_response).get("output_tokens"),
            },
        ]
    )
    display(comparison_df)
    preview_text("Native vision output", native_response.output_text)
    preview_text("Code Interpreter output", ci_response.output_text)
else:
    print(
        "Skipping the Code Interpreter comparison. Set RUN_OPTIONAL_CODE_INTERPRETER = True "
        "if you want to measure native vs tool-assisted latency on the police-form task."
    )


## 6. If you cannot use Code Interpreter, build a narrow crop-and-rerun pipeline

In restricted environments, you may not want to grant the model a general Python sandbox. A practical alternative is a two-stage workflow:

1. localize the field or region you care about
2. crop that region locally
3. rerun a smaller, more focused prompt on the crop

This often recovers much of the value of multi-pass inspection while keeping the control surface small.


In [ ]:
target_region = next(
    item for item in bbox_results if item["label"] == "vehicle_2_damaged_area_code"
)
crop_path = crop_from_normalized_bbox(
    "police_form",
    target_region["bbox"],
    padding_ratio=0.18,
)

display(Image.open(crop_path))

crop_prompt = """
Read the circled damaged-area code or codes in this cropped police-form region.

Return JSON with one key:
- damaged_area_codes: an array of strings

Return JSON only.
"""

crop_response = run_vision_request(
    crop_prompt,
    crop_path,
    model=PRIMARY_MODEL,
    detail="original",
)
crop_json = maybe_parse_json(crop_response.output_text)
if crop_json is None:
    raise ValueError(crop_response.output_text)

print(json.dumps(crop_json, indent=2))


## 7. Benchmark the exact setup you want to talk about publicly

If you want to make a launch splash, keep the asset, prompt, detail level, tool usage, and sampling settings fixed and only vary one factor at a time. The easiest way to mislead yourself is to compare outputs where the model, detail level, reasoning effort, and tool strategy all changed together.

The helper below is intentionally boring. It uses three known-answer tasks:

- handwritten form extraction
- floorplan reasoning
- invoice field extraction

It is opt-in because it can issue several API calls. By default it compares `gpt-5` against the current GPT-5.4 preview configured in `PRIMARY_MODEL`. If you want a wider launch table, set `OPENAI_MULTIMODAL_COMPARISON_MODELS` before running this notebook, for example:

```bash
export OPENAI_MULTIMODAL_COMPARISON_MODELS="gpt-5,gpt-5.2,gpt-5.4"
```


In [ ]:
invoice_eval_prompt = """
Read this hotel invoice and return JSON with these keys:
- hotel_name
- check_in_date
- check_out_date
- room_number
- total_gross_eur
- balance_due_eur
- breakfast_line_items_count

Rules:
- Normalize dates to YYYY-MM-DD.
- Return numeric fields as numbers, not strings.
- Return JSON only.
"""

benchmark_specs = [
    {
        "name": "handwriting_auto",
        "prompt": handwriting_prompt,
        "asset_ref": "handwritten_form",
        "expected": HANDWRITING_EXPECTED,
        "detail": "auto",
    },
    {
        "name": "handwriting_original",
        "prompt": handwriting_prompt,
        "asset_ref": "handwritten_form",
        "expected": HANDWRITING_EXPECTED,
        "detail": "original",
    },
    {
        "name": "floorplan_default_reasoning",
        "prompt": floorplan_prompt,
        "asset_ref": "floorplan",
        "expected": FLOORPLAN_EXPECTED,
        "detail": "original",
    },
    {
        "name": "floorplan_high_reasoning",
        "prompt": floorplan_prompt,
        "asset_ref": "floorplan",
        "expected": FLOORPLAN_EXPECTED,
        "detail": "original",
        "reasoning_effort": "high",
    },
    {
        "name": "invoice_structured",
        "prompt": invoice_eval_prompt,
        "asset_ref": "invoice",
        "expected": INVOICE_EXPECTED,
        "detail": "original",
    },
]

if RUN_OPTIONAL_BENCHMARKS:
    benchmark_rows = []
    for model in COMPARISON_MODELS:
        for spec in benchmark_specs:
            result = run_eval_case(
                spec["name"],
                spec["prompt"],
                spec["asset_ref"],
                spec["expected"],
                model=model,
                detail=spec.get("detail", "auto"),
                verbosity=spec.get("verbosity"),
                reasoning_effort=spec.get("reasoning_effort"),
            )
            benchmark_rows.append(
                {
                    key: result[key]
                    for key in [
                        "name",
                        "model",
                        "detail",
                        "verbosity",
                        "reasoning",
                        "elapsed_s",
                        "field_accuracy",
                        "matches",
                        "total_fields",
                        "output_tokens",
                    ]
                }
            )
    benchmark_df = pd.DataFrame(benchmark_rows).sort_values(["name", "model"])
    benchmark_summary = (
        benchmark_df.groupby("model", as_index=False)
        .agg(
            benchmark_cases=("name", "count"),
            matched_fields=("matches", "sum"),
            total_fields=("total_fields", "sum"),
            avg_elapsed_s=("elapsed_s", "mean"),
        )
        .assign(field_accuracy=lambda frame: (frame["matched_fields"] / frame["total_fields"]).round(3))
        .sort_values(["field_accuracy", "avg_elapsed_s"], ascending=[False, True])
    )
    display(benchmark_summary)
    display(benchmark_df)
else:
    print(
        "Skipping the benchmark suite. Set RUN_OPTIONAL_BENCHMARKS = True "
        "to run known-answer eval cases across the configured models."
    )


## Distilled guidance

- Start with native vision and `detail="auto"` for ordinary pages.
- Move to `detail="original"` for dense scans, handwriting, screenshots, and anything with tiny labels.
- Keep any sampling controls your model exposes fixed when you benchmark; model surfaces differ.
- Turn verbosity high only when you need faithful transcription. Otherwise, let the model stay concise.
- For region-level workflows, ask for normalized boxes and keep the coordinate system stable.
- Add `reasoning={"effort": "high"}` when the answer depends on multiple regions of a chart, table, form, floorplan, or technical diagram.
- Use Code Interpreter when a human would need to zoom, crop, rotate, or inspect several subregions before answering.
- If Code Interpreter is not available, emulate the same pattern with a narrow localize -> crop -> rerun workflow.
- Before making launch claims, benchmark the exact asset, prompt, detail level, tool configuration, and any exposed sampling settings you plan to publish.

## What is now explicit in this notebook

- `detail="original"` is tested on small email and phone fields, not just larger handwritten labels.
- `verbosity="high"` is tied to strict invoice-anchor coverage instead of vague transcription advice.
- `reasoning={"effort": "high"}` is shown as a tradeoff knob, not an unconditional upgrade.
- The dense-layout story now includes a tournament bracket example alongside the chart, floorplan, and form cases.
- The technical-drawing story now includes both a floorplan-style building plan and a UML-style developer diagram.
- Code Interpreter is framed as a multi-pass inspection tool with an explicit latency tradeoff.
- The model-comparison story is a real bundled-asset slice with measured results, not placeholder launch language.
